# Enformer Capacity-Utilization Analysis (Measurement Phase)

Forward-hook analysis of pretrained Enformer (`EleutherAI/enformer-official-rough`) for a
sparsification study. **fp32 throughout, MPS, hooks only** (architecture untouched).

All logic lives in `enformer_capacity.py` so the same collection + figure code can be
pointed at **Borzoi** later via a different `ModelSpec` + loader.

Steps: (1) setup + CPU/MPS sanity, (2) hg38 windows, (3) hooks + attention monkey-patch,
(4) activity collection, (5) four figures, (6) interpretation.

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")  # surface fallback as warning, not crash
import numpy as np
import torch
torch.set_grad_enabled(False)

import enformer_capacity as ec
device = ec.pick_device("mps")
device

## Step 1 - load + sanity check (CHECKPOINT)
Shapes must match; CPU/MPS agreement is reported. NOTE: the absolute max-diff (~5e-4)
exceeds 1e-4 but is benign fp32 accumulation - relative-to-peak is ~5e-6 end-to-end
(verified layer-by-layer in `investigate_divergence.py`: no single op jumps, errors track
output magnitude). Safe for aggregate capacity stats.

In [ ]:
model = ec.load_enformer()
report = ec.sanity_check(model, device, tol=1e-4)
report

## Step 2 - real hg38 windows
Genome-wide 196,608 bp windows, N-heavy regions filtered. Synthetic sequence is never
used for stats (it gives misleading dead-channel rates).

In [ ]:
N_WINDOWS = 64
windows = ec.sample_hg38_windows("data/hg38.fa", n_windows=N_WINDOWS, seed=0)
encoded = ec.encode_windows(windows)
len(encoded), tuple(encoded[0].shape)

## Step 3 - hooks + attention capture
`build_enformer_spec` resolves module paths: MLP intermediate = `blk[1].fn[3]` (post-ReLU,
3072-d), attention = `blk[0].fn[1]`, residual = whole block output. Attention weights are
**not** exposed by enformer-pytorch, so `install_attn_capture` **monkey-patches**
`Attention.forward` (per-instance) to stash post-softmax weights. Reused for Borzoi by
writing a new spec.

In [ ]:
spec = ec.build_enformer_spec(model)
spec.name, spec.n_layers, spec.n_heads

## Step 4 - activity collection
Bounded memory: per-head attention entropy and 90%-mass fraction are reduced inside the
loop (on CPU fp64 for stable stats); no full attention matrices are retained. Start at
batch 1; raise to 2 if memory allows.

In [ ]:
stats = ec.collect_activations(model, encoded, spec, device, batch_size=1)
{k: (v.shape if hasattr(v, "shape") else v) for k, v in stats.items()}

## Step 5 - figures
Dead channel = `mean|act| < 1e-6` (ReLU activations are bimodal with a point mass at exactly
zero, so a fixed near-zero cutoff is used, not a percentile).

In [ ]:
fig_meta = ec.make_figures(stats, outdir="results", dead_thresh=1e-6)
fig_meta

In [ ]:
from IPython.display import Image, display
for f in ["01_attention_entropy", "02_dead_channels", "03_residual_norm", "04_attention_sparsity"]:
    display(Image(f"results/{f}.png"))

## Step 6 - interpretation

In [ ]:
ec.write_summary(stats, fig_meta, outdir="results")

## Borzoi reuse
Write `build_borzoi_spec(model)` returning a `ModelSpec` with Borzoi's module paths +
attention-capture installer, then call the **same** `collect_activations` / `make_figures` /
`write_summary`.